## Data Ingestion

### document datastructure

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is the content of the document, I am using to create RAG system.",
    metadata={
        "source": "example.txt",
        "pages": 1,
        "author": "Kabir Khan",
        "date_created":"2026-04-22"
        }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Kabir Khan', 'date_created': '2026-04-22'}, page_content='This is the content of the document, I am using to create RAG system.')

In [3]:
### create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

### Python Txt File

In [4]:
import os

sample_text = {
    "../data/text_files/python_intro.txt": """Python is a high-level, interpreted programming language...""",
}

for filepath, content in sample_text.items():
    # Extract the directory path (e.g., "data/text_files/")
    folder_path = os.path.dirname(filepath)
    
    # Create the directory if it doesn't exist
    if folder_path:
        os.makedirs(folder_path, exist_ok=True)
        
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


### Machine Learning Txt File

In [5]:
import os

sample_text = {
    "../data/text_files/machine_learning.txt": """Machine learning is a subset of artificial intelligence...""",
}

for filepath, content in sample_text.items():
    # Extract the directory path (e.g., "data/text_files/")
    folder_path = os.path.dirname(filepath)
    
    # Create the directory if it doesn't exist
    if folder_path:
        os.makedirs(folder_path, exist_ok=True)
        
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)
print("Sample text files created successfully.")


Sample text files created successfully.


In [6]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
documents = loader.load()
print(documents)

c:\Users\Admin\Desktop\RAG_system_working\rag_system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming language...')]


In [7]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

### Load all .txt files in the specified directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

document = dir_loader.load()
document

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine learning is a subset of artificial intelligence...'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python is a high-level, interpreted programming language...')]

In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® Publisher for Microsoft 365', 'creator': 'Microsoft® Publisher for Microsoft 365', 'creationdate': '2026-04-21T14:58:35+05:00', 'source': '..\\data\\pdf\\example.pdf', 'file_path': '..\\data\\pdf\\example.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Research', 'subject': '', 'keywords': '', 'moddate': '2026-04-21T14:58:35+05:00', 'trapped': '', 'modDate': "D:20260421145835+05'00'", 'creationDate': "D:20260421145835+05'00'", 'page': 0}, page_content="AKD Securities Limited \nAKD RESEARCH \n• \nWe expect earnings for the AKD power universe to grow during 3QFY26E, primari-\nly due to i) surge in subsidiary profits, and ii) lower finance costs. \n• \nWe anticipate HUBC to announce an interim cash payout of PkR5.0/sh during \n3QFY26E, while no dividend is expected from NPL during the quarter.  \n• \nAt current levels, the revised TP of PkR215/sh for Dec’26 culminates in a ’HOLD’ \nstance on HUBC, while we maintain our previ

In [9]:
type(pdf_documents[0])

langchain_core.documents.base.Document

## RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [10]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [11]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    '''Process all pdf in the directroy'''
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.rglob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            #Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")   
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nTotal documents processed: {len(all_documents)}")
    return all_documents

#Process all PDF's in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")
            


Found 2 PDF files to process

Processing: example.pdf
 Loaded 5 pages

Processing: file-sample.pdf
 Loaded 4 pages

Total documents processed: 9


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Publisher for Microsoft 365', 'creator': 'Microsoft® Publisher for Microsoft 365', 'creationdate': '2026-04-21T14:58:35+05:00', 'source': '..\\data\\pdf\\example.pdf', 'file_path': '..\\data\\pdf\\example.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Research', 'subject': '', 'keywords': '', 'moddate': '2026-04-21T14:58:35+05:00', 'trapped': '', 'modDate': "D:20260421145835+05'00'", 'creationDate': "D:20260421145835+05'00'", 'page': 0, 'source_file': 'example.pdf', 'file_type': 'pdf'}, page_content="AKD Securities Limited \nAKD RESEARCH \n• \nWe expect earnings for the AKD power universe to grow during 3QFY26E, primari-\nly due to i) surge in subsidiary profits, and ii) lower finance costs. \n• \nWe anticipate HUBC to announce an interim cash payout of PkR5.0/sh during \n3QFY26E, while no dividend is expected from NPL during the quarter.  \n• \nAt current levels, the revised TP of PkR215/sh for Dec’26 culminates in a ’H

In [13]:
### Text Splitting get into Chucks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    '''Spliting Documents into smaller chunks'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show a example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [14]:
chunks= split_documents(all_pdf_documents)
chunks

Split 9 documents into 30 chunks

Example chunk:
Content: AKD Securities Limited 
AKD RESEARCH 
• 
We expect earnings for the AKD power universe to grow during 3QFY26E, primari-
ly due to i) surge in subsidiary profits, and ii) lower finance costs. 
• 
We an...
Metadata: {'producer': 'Microsoft® Publisher for Microsoft 365', 'creator': 'Microsoft® Publisher for Microsoft 365', 'creationdate': '2026-04-21T14:58:35+05:00', 'source': '..\\data\\pdf\\example.pdf', 'file_path': '..\\data\\pdf\\example.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Research', 'subject': '', 'keywords': '', 'moddate': '2026-04-21T14:58:35+05:00', 'trapped': '', 'modDate': "D:20260421145835+05'00'", 'creationDate': "D:20260421145835+05'00'", 'page': 0, 'source_file': 'example.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Publisher for Microsoft 365', 'creator': 'Microsoft® Publisher for Microsoft 365', 'creationdate': '2026-04-21T14:58:35+05:00', 'source': '..\\data\\pdf\\example.pdf', 'file_path': '..\\data\\pdf\\example.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Research', 'subject': '', 'keywords': '', 'moddate': '2026-04-21T14:58:35+05:00', 'trapped': '', 'modDate': "D:20260421145835+05'00'", 'creationDate': "D:20260421145835+05'00'", 'page': 0, 'source_file': 'example.pdf', 'file_type': 'pdf'}, page_content="AKD Securities Limited \nAKD RESEARCH \n• \nWe expect earnings for the AKD power universe to grow during 3QFY26E, primari-\nly due to i) surge in subsidiary profits, and ii) lower finance costs. \n• \nWe anticipate HUBC to announce an interim cash payout of PkR5.0/sh during \n3QFY26E, while no dividend is expected from NPL during the quarter.  \n• \nAt current levels, the revised TP of PkR215/sh for Dec’26 culminates in a ’H

### Embedding and Vector DB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [16]:
class EmbeddingManager:
    """Handles document embedding generation using SentanceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager
        Args: 
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of text
        
        Args: 
            texts: List of text strings to embed
            
        Returns: 
            numpy array of embeddings with shape (len(text), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embedding with shape: {embeddings.shape}")
        return embeddings
    
## Initialize the embedding manager
embedding_manager=EmbeddingManager() 
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\Admin\Desktop\RAG_system_working\rag_system\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Model loaded sucessfully. Embedding dimension: 384


### Vector Store

In [17]:
class VectorStore:
    """Manage document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._intialize_store()
        
    def _intialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddign for RAG"}
            )
            print(f"Vector store initialize. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self , documents:List[Any], embeddings: np.ndarray):
        """
        Add documents and their embedding to the vector store
        
        Args:
            documents: List of Langchain documents
            embedding: Correspondings embedding for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of document must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = [] 
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            #Document Content
            documents_text.append(doc.page_content)
            
            # Embedding 
            embeddings_list.append(embedding.tolist())
        
        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vectorstore=VectorStore()
vectorstore

Vector store initialize. Collection: pdf_documents
Existing documents in collection: 0


In [18]:
chunks

[Document(metadata={'producer': 'Microsoft® Publisher for Microsoft 365', 'creator': 'Microsoft® Publisher for Microsoft 365', 'creationdate': '2026-04-21T14:58:35+05:00', 'source': '..\\data\\pdf\\example.pdf', 'file_path': '..\\data\\pdf\\example.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': '', 'author': 'Research', 'subject': '', 'keywords': '', 'moddate': '2026-04-21T14:58:35+05:00', 'trapped': '', 'modDate': "D:20260421145835+05'00'", 'creationDate': "D:20260421145835+05'00'", 'page': 0, 'source_file': 'example.pdf', 'file_type': 'pdf'}, page_content="AKD Securities Limited \nAKD RESEARCH \n• \nWe expect earnings for the AKD power universe to grow during 3QFY26E, primari-\nly due to i) surge in subsidiary profits, and ii) lower finance costs. \n• \nWe anticipate HUBC to announce an interim cash payout of PkR5.0/sh during \n3QFY26E, while no dividend is expected from NPL during the quarter.  \n• \nAt current levels, the revised TP of PkR215/sh for Dec’26 culminates in a ’H

In [20]:
### Convert the text into embeddings
texts = [doc.page_content for doc in chunks]
texts

## Generate the Embeddings

embeddings = embedding_manager.generate_embedding(texts)

## Store into Vector Database
vectorstore.add_documents(chunks, embeddings)

Generating embedding for 30 texts..


Batches: 100%|██████████| 1/1 [00:08<00:00,  8.46s/it]

Generated Embedding with shape: (30, 384)
Adding 30 documents to vector store...
Successfully added 30 documents to vector store
Total documents in collection: 30
